In [2]:
# Read-only environment check (installs/changes nothing)
import platform, sys, os, subprocess, shutil, importlib, json
def sh(c):
    try: return subprocess.run(c, shell=True, capture_output=True, text=True, timeout=30).stdout.strip()
    except Exception as e: return f"ERR {e}"
print("arch:", platform.machine(), "| os:", platform.platform())
print("python:", sys.version.split()[0], "| exe:", sys.executable)
print("cpus:", os.cpu_count())
print(sh("free -g | head -2"))
print(sh("df -h ~ | tail -1"))
print("--- CPU / machine ---")
print(sh("grep -m1 'machine' /proc/cpuinfo; grep -m1 features /proc/cpuinfo"))
print(sh("cat /proc/sysinfo 2>/dev/null | grep -E 'Type|Model|LPAR Name|VM00 Control Program' | head"))
print("--- Secure Execution ---")
print("prot_virt_guest:", sh("cat /sys/firmware/uv/prot_virt_guest 2>/dev/null || echo 'n/a'"))
print("prot_virt_host:", sh("cat /sys/firmware/uv/prot_virt_host 2>/dev/null || echo 'n/a'"))
print("--- Crypto ---")
print(sh("ls /dev/z90crypt /dev/prandom 2>&1; lszcrypt 2>&1 | head -5"))
print(sh("grep -c . /proc/crypto; grep -E 'name.*(aes|gcm|sha)' /proc/crypto | sort -u | head -12"))
print("--- Packages ---")
for m in ["torch","torchvision","onnx","onnxruntime","tensorflow","sklearn","numpy","pandas","PIL","pydicom","cryptography","matplotlib","torchxrayvision","zdnn","ibm_zdnn_plugin"]:
    try:
        mod = importlib.import_module(m); print(f"  {m:18s}", getattr(mod,"__version__","ok"))
    except Exception as e: print(f"  {m:18s} MISSING ({type(e).__name__})")
print("--- tools ---")
for t in ["pip","conda","micromamba","docker","podman","zdlc","git","openssl","pvattest","genprotimg","kaggle"]:
    print(f"  {t:12s}", shutil.which(t))
print("--- network ---")
print(sh("curl -sS -o /dev/null -w '%{http_code}' --max-time 8 https://pypi.org/simple/ ; echo; curl -sS -o /dev/null -w '%{http_code}' --max-time 8 https://github.com ; echo; curl -sS -o /dev/null -w '%{http_code}' --max-time 8 https://huggingface.co"))
try:
    import torch; print("torch threads:", torch.get_num_threads(), "| backends:", [b for b in dir(torch.backends) if not b.startswith('_')][:20])
except Exception: pass

arch: s390x | os: Linux-5.14.0-687.58.1.el9_8.s390x-s390x-with-glibc2.39
python: 3.12.15 | exe: /opt/venv/bin/python3
cpus: 3

overlay          50G   12G   36G  25% /
--- CPU / machine ---
processor 0: version = FF,  identification = 07DFD8,  machine = 9175
features	: esan3 zarch stfle msa ldisp eimm dfp edat etf3eh highgprs te vx vxd vxe gs vxe2 vxp sort dflt vxp2 nnpa sie
Type:                 9175
Model:                702              ME1             
Model Capacity:       702              00000582
Model Perm. Capacity: 702              00000582
Model Temp. Capacity: 702              00000582
Model Var. Capacity:  702              00000582
Type 1 Percentage:    0
Type 2 Percentage:    0
Type 3 Percentage:    0
Type 4 Percentage:    0
--- Secure Execution ---
prot_virt_guest: n/a
prot_virt_host: n/a
--- Crypto ---
ls: cannot access '/dev/z90crypt': No such file or directory
ls: cannot access '/dev/prandom': No such file or directory
/bin/sh: line 1: lszcrypt: command not found
1064


In [3]:
# Cell 2: memory, NNPA plugin, and pip DRY-RUN (nothing is installed)
import subprocess, importlib.metadata as md, glob, os
def sh(c, t=180):
    r = subprocess.run(c, shell=True, capture_output=True, text=True, timeout=t); return (r.stdout + r.stderr).strip()
print(sh("grep -E 'MemTotal|MemAvailable' /proc/meminfo"))
print(sh("cat /sys/fs/cgroup/memory.max 2>/dev/null"))
for p in ["ibm-zdnn-plugin","tf2onnx","onnx2tf","tensorflow-cpu","keras","seaborn","scikit-learn"]:
    try: print(f"{p:18s}", md.version(p))
    except Exception: print(f"{p:18s} not installed")
print("zdnn .so:", glob.glob("/opt/venv/lib/python3.12/site-packages/**/_z_ops*.so", recursive=True)[:3])
os.environ['TF_CPP_MIN_LOG_LEVEL']='2'
import tensorflow as tf
print("TF devices:", tf.config.list_physical_devices())
print("--- pip dry-run ---")
for pkg in ["torchvision","onnxruntime","cryptography","pydicom","torchxrayvision","kaggle","scikit-image"]:
    out = sh(f"pip install --dry-run --no-deps --quiet --report - {pkg} 2>&1 | python3 -c \"import sys,json;d=sys.stdin.read();\ntry:\n j=json.loads(d[d.index('{'):]);print([ (i['metadata']['name'],i['metadata']['version'],i['download_info']['url'].split('/')[-1]) for i in j['install']])\nexcept Exception as e: print(d[-300:])\"")
    print(f"{pkg:16s} -> {out}")

SyntaxError: invalid syntax. Perhaps you forgot a comma? (317436520.py, line 16)